# KKT analysis: exogenous P2P price, shadow prices, and individual rationality
### Hardcoded experiment — January 15, 2016 · Base structure

Two versions of the centralized solve are compared:

| | **Standard** | **Clean** |
|---|---|---|
| c_P (trade regularisation) | 1e-3 EUR/kW²·h | 0 |
| c_Q (reactive regularisation) | 1e-4 EUR/kVAR²·h | 0 |
| trade_fee | 0.001 EUR/kWh | 0 |

The clean version removes all terms that create a wedge between buyer and
seller KKT conditions, leaving the shadow price of the reciprocity constraint
equal to the exogenous P2P price in theory. We then check whether this is
actually the case, and whether removing these terms changes P1's savings.


## 1  KKT conditions and shadow price derivation

The centralised problem is:

$$\min_{x \in \mathcal{X}} \sum_i J_i(x) \quad \text{s.t.}\quad q_{ij,t} + q_{ji,t} = 0 \;\forall (i,j)\in\mathcal{E}, t\in\mathcal{T}$$

Attaching multiplier $\mu_{ij,t}$ to each reciprocity constraint, the KKT
stationarity condition for $q_{ij,t}$ is:

$$\frac{\partial}{\partial q_{ij,t}}\left[\sum_k J_k(x)\right] + \mu_{ij,t} = 0$$

### With the full cost (standard version)

$J_i$ contains three terms in $q_{ij,t}$ (assuming $q_{ij,t}>0$, i.e. $i$ sells):

$$\frac{\partial J_i}{\partial q_{ij,t}} = \Delta t\left(-\lambda^{p2p}_t + f^{fee} + 2c_P q_{ij,t}\right)$$

For prosumer $j$ ($q_{ji,t} = -q_{ij,t} < 0$, so $j$ buys):

$$\frac{\partial J_j}{\partial q_{ji,t}} = \Delta t\left(-\lambda^{p2p}_t - f^{fee} - 2c_P q_{ij,t}\right)$$

Both must equal $-\mu_{ij,t}$, giving **two different expressions for the
same dual variable**:

$$\mu_{ij,t} = \Delta t(\lambda^{p2p}_t - f^{fee} - 2c_P q^*_{ij,t}) \quad\text{(from seller)}$$
$$\mu_{ij,t} = \Delta t(\lambda^{p2p}_t + f^{fee} + 2c_P q^*_{ij,t}) \quad\text{(from buyer)}$$

These are **inconsistent** unless $f^{fee}=0$ and $c_P=0$ on trade variables.
In the standard version Gurobi finds a primal-dual pair that satisfies both
to within the OptimalityTol, but the shadow price is not a clean economic signal.

### With c_P = 0, c_Q = 0, f^{fee} = 0 (clean version)

Both conditions collapse to:

$$\mu^*_{ij,t} = \Delta t \lambda^{p2p}_t$$

The shadow price of the reciprocity constraint **equals the exogenous P2P
price exactly**. The dual structure is clean and consistent.

### Does this guarantee individual rationality?

**No.** The condition $\mu^* = \Delta t\lambda^{p2p}$ tells us the game is
well-posed — buyer and seller agree on the marginal value of the trade.
It does not tell us whether the forced trade schedule makes prosumer $i$
better off than the no-trade baseline.

IR holds for prosumer $i$ if and only if:

$$J_i(x^{SO}) \leq J_i^{nt}$$

Equivalently, the P2P revenue received by $i$ must cover the **extra device
costs** induced by the socially-optimal trade schedule. For P1 specifically:

$$\Delta t \sum_t \lambda^{p2p}_t q^*_{1j,t}
  \geq \underbrace{\Delta t\, c^b \sum_t (c_{1,t}^* - c_{1,t}^{nt}) + (d_{1,t}^* - d_{1,t}^{nt})}_{\text{extra BESS cycling}}
     + \underbrace{\Delta t\, c^{ev} \sum_t (\cdot)}_{\text{extra EV cycling}}$$

If the left side (P2P revenue) is less than the right side (extra cycling
costs forced by the social optimum), P1 is worse off trading — regardless
of whether the dual structure is clean.

**This is the key claim for your thesis**: the failure of IR is not a modelling
artefact from the regularisation, but an inherent consequence of fixing
$\lambda^{p2p}$ exogenously at a level that does not compensate P1 for the
flexibility extracted by the social optimum.

The experiment below tests this by removing the regularisation and checking
whether P1's savings change.


## 2  Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from scenarios import make_scenario_date
from prosumer  import add_prosumer
from metrics   import savings, sw, tw, mm, no_trade_costs

plt.rcParams.update({'font.family':'sans-serif',
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.25,'figure.dpi':120})

DATE = '2016-01-15'
data = make_scenario_date(DATE, seed=42, resolution_h=0.5)
N, T, dt = len(data['demand']), len(data['buy_price']), data['dt']

print(f"c_P  = {data.get('p_cost',0):.1e}  EUR/kW²·h")
print(f"c_Q  = {data.get('q_cost',0):.1e}  EUR/kVAR²·h")
print(f"fee  = {data['trade_fee']:.4f}  EUR/kWh")
print(f"p2p  range: [{min(data['p2p_price']):.3f}, {max(data['p2p_price']):.3f}] EUR/kWh")

# No-trade baseline (same for both versions)
RESULT_DIR = ROOT / 'structures' / 'base' / 'results' / DATE
nt   = json.loads((RESULT_DIR / 'no_trade.json').read_text())
J_nt = np.array(nt['J_nt'])
print(f"\nNo-trade costs: {J_nt}")

names = [p['name'] for p in [
    {'name':'Res. ++'},{'name':'Office'},
    {'name':'Res. HP'},{'name':'Res. PV'}]]


## 3  Centralised solver with dual variable extraction

In [ ]:
def centralized_with_duals(data):
    """Run centralised solve and return costs, trades, and dual variables
    on the reciprocity constraints. Hardcoded inline — no changes to algo files."""
    m = gp.Model('cent_duals')
    m.Params.OutputFlag     = 0
    m.Params.FeasibilityTol = 1e-9
    m.Params.OptimalityTol  = 1e-9

    agents = [add_prosumer(m, i, data) for i in range(N)]

    # Named reciprocity constraints so we can read their dual (.Pi)
    recip = {}
    for i, j in data['edges']:
        for t in range(T):
            recip[(i,j,t)] = m.addConstr(
                agents[i]['trade'][j,t] + agents[j]['trade'][i,t] == 0,
                name=f'recip_{i}_{j}_{t}')

    m.setObjective(gp.quicksum(a['cost'] for a in agents), gp.GRB.MINIMIZE)
    m.optimize()

    if m.Status not in (gp.GRB.OPTIMAL, gp.GRB.SUBOPTIMAL):
        raise RuntimeError(f"Status {m.Status}")

    costs  = np.array([a['cost'].getValue() for a in agents])
    duals  = {k: c.Pi for k, c in recip.items()}   # shadow price * dt per unit
    trades = {}
    for i, a in enumerate(agents):
        for (j, t), v in a['trade'].items():
            trades[(i,j,t)] = v.X

    m.dispose()
    return {'costs': costs, 'duals': duals, 'trades': trades}


## 4  Run both versions

In [ ]:
# Standard version — as configured in scenarios.py
print("Running: standard (c_P=1e-3, c_Q=1e-4, fee=0.001)...")
res_std = centralized_with_duals(data)
d_std   = savings({'costs': res_std['costs']}, J_nt)
print(f"  Total cost: {res_std['costs'].sum():.4f} EUR")

# Clean version — remove regularisation and trade fee
data_clean = {**data, 'p_cost': 0.0, 'q_cost': 0.0, 'trade_fee': 0.0}
print("Running: clean   (c_P=0, c_Q=0, fee=0)...")
res_cln = centralized_with_duals(data_clean)
d_cln   = savings({'costs': res_cln['costs']}, J_nt)
print(f"  Total cost: {res_cln['costs'].sum():.4f} EUR")

print(f"\n{'':25s} {'Standard':>10} {'Clean':>10}")
print('─'*47)
for i in range(N):
    print(f"  P{i+1} {names[i]:20s} {d_std[i]:>+10.4f} {d_cln[i]:>+10.4f}")
print(f"  {'SW = Σdᵢ':25s} {d_std.sum():>+10.4f} {d_cln.sum():>+10.4f}")
print(f"  {'MM = min dᵢ':25s} {d_std.min():>+10.4f} {d_cln.min():>+10.4f}")
print(f"  {'TW (raw costs)':25s} {tw(J_nt, res_std['costs']):>+10.4f} {tw(J_nt, res_cln['costs']):>+10.4f}")


## 5  Savings comparison

In [ ]:
x = np.arange(N); w = 0.32
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x-w/2, d_std, w, label='Standard (c_P, c_Q, fee)',
       color='#42A5F5', edgecolor='white')
ax.bar(x+w/2, d_cln, w, label='Clean (c_P=c_Q=fee=0)',
       color='#FFA726', edgecolor='white', hatch='///')
ax.axhline(0, color='black', lw=1)
ax.set_xticks(x)
ax.set_xticklabels([f'P{i+1}\n{names[i]}' for i in range(N)], fontsize=9)
ax.set_ylabel('Savings fraction $d_i$')
ax.set_title(f'Savings at social optimum — standard vs clean — {DATE}',
             fontweight='bold')
ax.legend(fontsize=9)
for bars, d_vec in [(ax.patches[:N], d_std), (ax.patches[N:], d_cln)]:
    pass  # annotations below

for xi, (vs, vc) in enumerate(zip(d_std, d_cln)):
    for val, offset in [(vs, -w/2), (vc, w/2)]:
        va = 'bottom' if val >= 0 else 'top'
        ax.text(xi+offset, val+(0.01 if val>=0 else -0.01), f'{val:+.3f}',
                ha='center', va=va, fontsize=8)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'kkt_savings_comparison.pdf', bbox_inches='tight')
plt.show()


## 6  Shadow price vs exogenous P2P price

The dual variable of constraint $q_{ij,t}+q_{ji,t}=0$ is $\mu_{ij,t}$.
Dividing by $\Delta t$ gives the shadow price in EUR/kWh — the marginal
value of relaxing the reciprocity constraint by 1 kW for one timestep.

In the clean version, theory predicts $\mu^*_{ij,t}/\Delta t = \lambda^{p2p}_t$.
We check this empirically.


In [ ]:
p2p = np.array(data['p2p_price'])
hours = np.arange(T) * dt

# Pick one edge for illustration
edge = data['edges'][0]   # e.g. (0,1)
i0, j0 = edge

mu_std = np.array([res_std['duals'][(i0,j0,t)] / dt for t in range(T)])
mu_cln = np.array([res_cln['duals'][(i0,j0,t)] / dt for t in range(T)])

fig, axes = plt.subplots(1, 2, figsize=(13,4), sharey=False)

for ax, mu, title in [
    (axes[0], mu_std, f'Standard — edge P{i0+1}→P{j0+1}'),
    (axes[1], mu_cln, f'Clean — edge P{i0+1}→P{j0+1}'),
]:
    ax.plot(hours, p2p, 'k--', lw=1.5, label='P2P price $\lambda^{p2p}$')
    ax.plot(hours, mu,  color='#EF5350', lw=2, label='Shadow price $\mu^*/\Delta t$')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('EUR/kWh')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Shadow price of reciprocity constraint vs exogenous P2P price',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'kkt_shadow_prices.pdf', bbox_inches='tight')
plt.show()

print("\nMax deviation |shadow - p2p|:")
print(f"  Standard: {np.max(np.abs(mu_std - p2p)):.6f} EUR/kWh")
print(f"  Clean:    {np.max(np.abs(mu_cln - p2p)):.6f} EUR/kWh")


## 7  Trade volumes

In [ ]:
fig, axes = plt.subplots(N, 1, figsize=(12,8), sharex=True)
for i in range(N):
    ax = axes[i]
    peers = [b if a==i else a for a,b in data['edges'] if i in (a,b)]
    for j in peers:
        q_std = np.array([res_std['trades'].get((i,j,t),0.) for t in range(T)])
        q_cln = np.array([res_cln['trades'].get((i,j,t),0.) for t in range(T)])
        ax.plot(hours, q_std, '-',  lw=1.5, label=f'Std P{i+1}→P{j+1}')
        ax.plot(hours, q_cln, '--', lw=1.5, label=f'Cln P{i+1}→P{j+1}')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel(f'P{i+1} kW', fontsize=8)
    if i == 0:
        ax.legend(fontsize=7, ncol=4, loc='upper right')
axes[-1].set_xlabel('Hour of day')
plt.suptitle('Trade flows: standard vs clean', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'kkt_trade_flows.pdf', bbox_inches='tight')
plt.show()

# Net export per prosumer
print("Net P2P export (kWh):")
print(f"  {'':20s} {'Standard':>10} {'Clean':>10}")
print('─'*42)
for i in range(N):
    peers = [b if a==i else a for a,b in data['edges'] if i in (a,b)]
    net_s = sum(res_std['trades'].get((i,j,t),0.) for j in peers for t in range(T)) * dt
    net_c = sum(res_cln['trades'].get((i,j,t),0.) for j in peers for t in range(T)) * dt
    print(f"  P{i+1} {names[i]:18s} {net_s:>+10.2f} {net_c:>+10.2f}")


## 8  Individual rationality decomposition

For each prosumer, decompose the cost change $J_i^{SO} - J_i^{nt}$ into
contributions from P2P revenue, BESS cycling, and EV cycling.
A positive total means the prosumer is worse off trading.


In [ ]:
def decompose(res, data, J_nt):
    """Break cost change into components for each prosumer."""
    bp = np.array(data['buy_price']); sp = np.array(data['sell_price'])
    pp = np.array(data['p2p_price']); fee = data['trade_fee']
    c_b = data['battery_cost'];       c_ev = data.get('ev_cost',0.)
    rows = []
    for i in range(N):
        peers = [b if a==i else a for a,b in data['edges'] if i in (a,b)]
        net_q = np.array([sum(res['trades'].get((i,j,t),0.) for j in peers)
                          for t in range(T)])
        abs_q = np.array([sum(abs(res['trades'].get((i,j,t),0.)) for j in peers)
                          for t in range(T)])

        p2p_rev  =  dt * float(np.sum(pp * net_q))
        trade_fee_cost = dt * fee * float(np.sum(abs_q))
        delta_J  = float(res['costs'][i]) - float(J_nt[i])
        rows.append({'prosumer': f'P{i+1} {names[i]}',
                     'ΔJ total': delta_J,
                     'P2P revenue': -p2p_rev,
                     'trade fee': trade_fee_cost,
                     'other (device)': delta_J + p2p_rev - trade_fee_cost})
    return rows

import pandas as pd
print("Standard:")
df_s = pd.DataFrame(decompose(res_std, data, J_nt)).set_index('prosumer')
print(df_s.round(4).to_string())
print()
print("Clean:")
df_c = pd.DataFrame(decompose(res_cln, data_clean, J_nt)).set_index('prosumer')
print(df_c.round(4).to_string())


## 9  What this tells us

After running this notebook, check three things:

**Shadow price alignment.** In the clean version, the plot in Section 6
should show the shadow price tracking the P2P price closely (small deviation).
In the standard version, the regularisation and trade fee create a wedge.
This is the empirical confirmation of the theoretical derivation.

**P1's savings with clean parameters.** If d_1 is still negative in the
clean version, it proves conclusively that the negative saving is not caused
by the regularisation or trade fee — it is an inherent property of the
game with this P2P price level and prosumer heterogeneity.

**Trade volume change.** Without regularisation the solver may find different
trade flows (the optimal face is flat so multiple solutions exist).
Compare net exports to see how much the regularisation was shaping the
outcome versus the economic signals.

The key theoretical statement for the thesis is then:

> Even with a clean dual structure (c_P = c_Q = f^{fee} = 0), the
> shadow price $\mu^*_{ij,t}/\Delta t = \lambda^{p2p}_t$ — but P1
> remains worse off trading because the exogenous P2P price does not
> cover P1's marginal cost of providing flexibility to the community.
> This is an inherent consequence of fixing the P2P price exogenously
> rather than deriving it from the equilibrium dual variable.
